In [9]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import java.math.BigDecimal
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}

enum class Context { ELIMINATION, BUDGET, CLUSTERING, FINAL }

val percentageFraction = 1
val colsWithoutPercentages = ""
val gradient = 0.1
val useRefTable = true
val budgetFactor = "1.00"
val onlyTabular = true

val mode = Mode.FLAT
val algorithm = Algorithm.OP
val context = Context.BUDGET
val budgetSuffix = "all"

val fileName = when (context) {
    Context.ELIMINATION -> "comparison_elimination_${mode.name.lowercase()}.csv"
    Context.BUDGET -> "budget_comparison_$budgetSuffix.csv"
    Context.CLUSTERING -> "comparison_clustering_${mode.name.lowercase()}.csv"
    Context.FINAL -> "comparison_final_avg_${mode.name.lowercase()}.csv"
}

val relativePath = when (context) {
    Context.ELIMINATION -> "/op-solver-strict/results/final/elimination/${mode.name.lowercase()}/"
    Context.BUDGET -> "/op-solver-strict/results/budget/comparison-${mode.name.lowercase()}/"
    Context.CLUSTERING -> "/op-solver-strict/results/final/clustering/${mode.name.lowercase()}/"
    Context.FINAL -> "/op-solver-strict/results/final/"
}

    //"/op-solver-strict/results/final/" //${context.name.lowercase()}/comparison/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df

name,eqc,elc,csml,cdcmx
eil101,59,60,59,59
gil262,132,141,129,133
pr299,148,150,147,141
lin318,183,187,184,183
rd400,202,195,199,190
d493,297,303,304,283
u574,305,309,313,300
u724,385,381,384,375
pcb1173,571,576,570,567
fl1400,910,935,966,661


In [10]:
val baselinePath = "/op-solver-strict/results/ref/${mode.name.lowercase()}_1.csv"
val pathToBaseLine = java.nio.file.Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath()
    .toString() + baselinePath
val orderedInstances = listOf("eil101", "gil262", "pr299", "lin318", "rd400", "d493", "u574", "u724", "pcb1173", "fl1400", "pr2392").map { if (it == "instance") it else it + "-gen3-50" }
val header = listOf("instance", "0.30", "0.40", "0.50","0.60","0.70","0.80")
var baselineDf = DataFrame.readCsv(pathToBaseLine)
baselineDf = baselineDf.sortWith(compareBy { row -> orderedInstances.indexOf(row["name"].toString()) })
    .reorderColumnsBy { colums -> header.indexOf(colums.name()) }
val colsToConvert = baselineDf.columnNames().drop(1)

baselineDf = colsToConvert.fold(baselineDf) { acc, col ->
    acc.convert(col) { v ->
        when (v) {
            is Number -> v.toInt()
            else -> v?.toString()?.trim()?.toInt()
                ?: throw IllegalArgumentException("Cannot parse column \$col value '\$v' to Int")
        }
    }
}
baselineDf

name,1.00,0.30,0.50,0.70
eil101,59,19,32,43
gil262,135,41,70,99
pr299,149,47,77,105
lin318,182,59,95,131
rd400,196,65,105,146
d493,304,66,147,204
u574,306,92,157,219
u724,382,110,196,278
pcb1173,574,176,293,409
fl1400,933,383,544,648


In [11]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Number).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[60, 141, 150, 187, 202, 304, 313, 385, 576, 966, 1200]

In [12]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = if (useRefTable) {
    bestValues.mapIndexed { index, resultMax ->
        max(resultMax, (baselineDf[budgetFactor.toString()][index] as Number).toInt())
    }
} else {
    bestValues.mapIndexed { index, resultMax -> max(resultMax, (df[colsWithoutPercentages][index] as Number).toInt()) }
}
rowMaxValues

[60, 141, 150, 187, 202, 304, 313, 385, 576, 966, 1200]

In [13]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Int>()
}.map { row -> row!!.toInt()}
rowMinValues

[59, 129, 141, 183, 190, 283, 300, 375, 567, 661, 1174]

In [14]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Int>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = if (useRefTable) {
        (baselineDf[budgetFactor.toString()][row.index()] as Number).toInt()
    } else {
        (df.get(colsWithoutPercentages)[row] as Number).toInt()
    }
    val percentage = calculatePercentage(refValue, compValue)
    return "${formatePercentage(percentage)}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = if (useRefTable) {
            (baselineDf[budgetFactor.toString()][row.index()] as Number).toInt()
        } else {
            (df.get(colsWithoutPercentages)[row] as Number).toInt()
        }
        calculatePercentage(refValue, (col[row] as Number).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it) + "&"
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -0.4\%&, +0.9\%&, +0.1\%&, -4.8\%&]

In [15]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} - & \\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} - & \\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Number).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} {$percentage}" +
                    "&\\cellcolor{cyan!$saturation}{\\textbf{$value*}}"
        }else {
            "\\cellcolor{cyan!$saturation} $percentage" +
                    "&\\cellcolor{cyan!$saturation}{$value}"
        }
    }
}
stringdf

name,eqc,elc,csml,cdcmx
eil101,\cellcolor{cyan!83} +0.0\%&\cellcolor...,\cellcolor{cyan!100} {+1.7\%}&\cellco...,\cellcolor{cyan!83} +0.0\%&\cellcolor...,\cellcolor{cyan!83} +0.0\%&\cellcolor...
gil262,\cellcolor{cyan!36} -2.2\%&\cellcolor...,\cellcolor{cyan!100} {+4.4\%}&\cellco...,\cellcolor{cyan!14} -4.4\%&\cellcolor...,\cellcolor{cyan!43} -1.5\%&\cellcolor...
pr299,\cellcolor{cyan!86} -0.7\%&\cellcolor...,\cellcolor{cyan!100} {+0.7\%}&\cellco...,\cellcolor{cyan!80} -1.3\%&\cellcolor...,\cellcolor{cyan!40} -5.4\%&\cellcolor...
lin318,\cellcolor{cyan!78} +0.5\%&\cellcolor...,\cellcolor{cyan!100} {+2.7\%}&\cellco...,\cellcolor{cyan!83} +1.1\%&\cellcolor...,\cellcolor{cyan!78} +0.5\%&\cellcolor...
rd400,\cellcolor{cyan!100} {+3.1\%}&\cellco...,\cellcolor{cyan!65} -0.5\%&\cellcolor...,\cellcolor{cyan!85} +1.5\%&\cellcolor...,\cellcolor{cyan!40} -3.1\%&\cellcolor...
d493,\cellcolor{cyan!76} -2.3\%&\cellcolor...,\cellcolor{cyan!96} -0.3\%&\cellcolor...,\cellcolor{cyan!100} {+0.0\%}&\cellco...,\cellcolor{cyan!30} -6.9\%&\cellcolor...
u574,\cellcolor{cyan!74} -0.3\%&\cellcolor...,\cellcolor{cyan!87} +1.0\%&\cellcolor...,\cellcolor{cyan!100} {+2.3\%}&\cellco...,\cellcolor{cyan!58} -2.0\%&\cellcolor...
u724,\cellcolor{cyan!100} {+0.8\%}&\cellco...,\cellcolor{cyan!89} -0.3\%&\cellcolor...,\cellcolor{cyan!97} +0.5\%&\cellcolor...,\cellcolor{cyan!74} -1.8\%&\cellcolor...
pcb1173,\cellcolor{cyan!91} -0.5\%&\cellcolor...,\cellcolor{cyan!100} {+0.3\%}&\cellco...,\cellcolor{cyan!89} -0.7\%&\cellcolor...,\cellcolor{cyan!84} -1.2\%&\cellcolor...
fl1400,\cellcolor{cyan!42} -2.5\%&\cellcolor...,\cellcolor{cyan!67} +0.2\%&\cellcolor...,\cellcolor{cyan!100} {+3.5\%}&\cellco...,\cellcolor{cyan!0} -29.2\%&\cellcolor...


In [16]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return if (onlyTabular) {
        """
        \begin{tabular}{ $formating  }
            \hline
            \multicolumn{$amountColumns}{|c|}{$title} \\
            \hline
                $header \\
            \hline
                $body
            \hline
        \end{tabular}
         """
    } else { """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """ }
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}


val amountColumns = (df.columns().size * 2 - 1).toString()
val formating = "|p{1.3cm}|" + List(df.columns().size-1) { "p{1.3cm} p{1.1cm}" }.joinToString("|") + "|"
val header = df.columnNames().first() + " & " +  df.columnNames()
    .filterIndexed{index, _ -> index != 0}.joinToString(separator = " & & ") + " &"

val label = "tab:elim:${mode.name.lowercase()}"
val title = "emimination methods ${mode.name.lowercase()}."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Emimination method comparison for \$R' = 0.5\$ and \$\\alpha = 0.25\$ with $\\gamma = 0.5\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


        \begin{tabular}{ |p{1.3cm}|p{1.3cm} p{1.1cm}|p{1.3cm} p{1.1cm}|p{1.3cm} p{1.1cm}|p{1.3cm} p{1.1cm}|  }
            \hline
            \multicolumn{9}{|c|}{emimination methods flat.} \\
            \hline
                name & eqc & & elc & & csml & & cdcmx & \\
            \hline
                eil101 & \cellcolor{cyan!83} +0.0\%&\cellcolor{cyan!83}{59} & \cellcolor{cyan!100} {+1.7\%}&\cellcolor{cyan!100}{\textbf{60*}} & \cellcolor{cyan!83} +0.0\%&\cellcolor{cyan!83}{59} & \cellcolor{cyan!83} +0.0\%&\cellcolor{cyan!83}{59} \\ 
gil262 & \cellcolor{cyan!36} -2.2\%&\cellcolor{cyan!36}{132} & \cellcolor{cyan!100} {+4.4\%}&\cellcolor{cyan!100}{\textbf{141*}} & \cellcolor{cyan!14} -4.4\%&\cellcolor{cyan!14}{129} & \cellcolor{cyan!43} -1.5\%&\cellcolor{cyan!43}{133} \\ 
pr299 & \cellcolor{cyan!86} -0.7\%&\cellcolor{cyan!86}{148} & \cellcolor{cyan!100} {+0.7\%}&\cellcolor{cyan!100}{\textbf{150*}} & \cellcolor{cyan!80} -1.3\%&\cellcolor{cyan!80}{147} & \cellcolor{cyan!40} -5.4\%&\cel